# Preprocesado y Feature Engineering

En este notebook transformamos el dataset saneado (`data_sanitized.csv`) de transacciones individuales en una **serie temporal diaria** lista para ser modelizada, enriqueciéndola mediante ingeniería de variables.

El notebook es **autosuficiente**: carga el CSV desde disco sin depender de ningún notebook previo.

Las etapas que se realizan son:
- **4.1** Agregación diaria y creación de la variable objetivo `Ventas`
- **4.2** Variables categóricas de resumen diario (producto top, país top, clientes únicos)
- **4.3** Extracción de variables temporales desde la fecha
- **4.4** Análisis y limpieza de las nuevas variables
- **4.6** Análisis de correlación y selección de variables
- **4.7** Lags (retrasos temporales)
- **4.8** Medias móviles
- **4.9** Eventos especiales
- **4.10** Escalado y exportación (train/test split + CSV)


## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual y definimos las rutas del proyecto.

A continuación cargamos `data_sanitized.csv` — el CSV limpio exportado por el notebook de limpieza — que ya incluye las columnas auxiliares `Fecha`, `Mes`, `DiaSemana`, `EsCancelacion` y `TotalPrice`, por lo que no es necesario recalcularlas.

Se inicializa también el diccionario de auditoría `stats_preprocesing` que registrará las transformaciones realizadas en este notebook.


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_INTERIM   = '../../../data/interim/'
RUTA_GRAFICOS  = '../../../graphics/'
RUTA_PROCESSED = '../../../data/processed/'
os.makedirs(RUTA_GRAFICOS,  exist_ok=True)
os.makedirs(RUTA_PROCESSED, exist_ok=True)

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


In [4]:
# Carga del CSV limpio (output del notebook de limpieza)
df_raw     = pd.read_csv(
    RUTA_INTERIM + 'data_sanitized.csv',
    parse_dates=['InvoiceDate', 'Fecha']
)
df_working = df_raw.copy()

# EsCancelacion puede llegar como string 'True'/'False' → forzar a bool
df_working['EsCancelacion'] = df_working['EsCancelacion'].astype(bool)

# Diccionario de auditoría (se actualiza en cada paso de transformación)
stats_preprocesing = {
    'Registros de entrada (transacciones)': len(df_raw),
}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 58)
print(f'  DATASET CARGADO  (data_sanitized.csv)')
print('=' * 58)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print(f'  Rango    : {df_raw["Fecha"].min().date()} → {df_raw["Fecha"].max().date()}')
print('=' * 58)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas heredadas del notebook de limpieza:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')
print(f'\n  Columnas completas: {list(df_working.columns)}')


  DATASET CARGADO  (data_sanitized.csv)
  Filas    :    531,172
  Columnas :         13
  Rango    : 2010-12-01 → 2011-12-09

  df_working activo : 531,172 filas
  Columnas heredadas del notebook de limpieza:
    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice

  Columnas completas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Fecha', 'Mes', 'DiaSemana', 'EsCancelacion', 'TotalPrice']


# 4. TRANSFORMACIÓN DE DATOS


### 4.1 Agregación diaria y creación de la variable objetivo "Ventas"

El dataset limpio tiene una fila por transacción de producto. Aquí lo convertimos en una **serie temporal diaria** con las siguientes variables:

| Variable | Descripción |
|---|---|
| `Ventas` | Venta **neta** diaria en £ — incluye cancelaciones (TotalPrice negativo) que restan automáticamente |
| `NumTransacc` | Número de líneas de producto vendidas ese día (solo ventas, sin cancelaciones) |
| `NumPedidos` | Número de pedidos únicos (`InvoiceNo`) ese día |
| `NumClientes` | Número de clientes únicos (`CustomerID`) ese día |
| `UnidadesVendidas` | Total de unidades físicas vendidas ese día |

Tras el `groupby`, se hace **reindex al rango completo** de fechas rellenando con 0 los días sin actividad (festivos, fines de semana cerrados). Esto es crítico para que los lags y medias móviles sean temporalmente consistentes: sin reindex, el Lag1 de un lunes sería el viernes, no el domingo.


In [5]:
# 4.1 — Agregación diaria y creación de la variable objetivo 'Ventas'
n_ventas   = (df_working['EsCancelacion'] == False).sum()
n_cancelac = (df_working['EsCancelacion'] == True).sum()
print(f"── 4.1 Agregación diaria ──")
print(f"  Filas en df_working   : {len(df_working):,}")
print(f"    → Ventas reales     : {n_ventas:,}")
print(f"    → Cancelaciones     : {n_cancelac:,}")
print(f"  Decisión: Ventas = venta NETA (cancelaciones restan como TotalPrice negativo)")

# ── Venta neta diaria (incluye cancelaciones) ─────────────────────────────────
ventas_netas = (
    df_working
    .groupby('Fecha', sort=True)
    .agg(Ventas=('TotalPrice', 'sum'))
    .reset_index()
)

# ── Variables de volumen (solo filas de venta, no cancelaciones) ──────────────
df_solo_ventas = df_working[df_working['EsCancelacion'] == False]

features_volumen = (
    df_solo_ventas
    .groupby('Fecha', sort=True)
    .agg(
        NumTransacc      = ('TotalPrice', 'count'),
        NumPedidos       = ('InvoiceNo',  'nunique'),
        NumClientes      = ('CustomerID', 'nunique'),
        UnidadesVendidas = ('Quantity',   'sum'),
    )
    .reset_index()
)

# ── Merge y reindex al rango completo de fechas ───────────────────────────────
df_daily = ventas_netas.merge(features_volumen, on='Fecha', how='left')

rango_completo = pd.date_range(
    start=df_daily['Fecha'].min(),
    end=df_daily['Fecha'].max(),
    freq='D'
)

df_daily = (
    df_daily
    .set_index('Fecha')
    .reindex(rango_completo)
    .rename_axis('Fecha')
    .fillna(0)
    .reset_index()
)

stats_preprocesing['Días serie temporal']     = len(df_daily)
stats_preprocesing['Días con ventas > 0']     = int((df_daily['Ventas'] > 0).sum())
stats_preprocesing['Días sin actividad (0)']  = int((df_daily['Ventas'] == 0).sum())


── 4.1 Agregación diaria ──
  Filas en df_working   : 531,172
    → Ventas reales     : 522,504
    → Cancelaciones     : 8,668
  Decisión: Ventas = venta NETA (cancelaciones restan como TotalPrice negativo)


In [6]:
# 4.1 — Verificación y resumen del dataframe diario
print(f"  Rango temporal             : {df_daily['Fecha'].min().date()} → {df_daily['Fecha'].max().date()}")
print(f"  Días en el rango completo  : {len(rango_completo)}")
print(f"  Días con Ventas > 0        : {(df_daily['Ventas'] > 0).sum()}")
print(f"  Días con Ventas = 0 (hueco): {(df_daily['Ventas'] == 0).sum()}")
print(f"  Días con Ventas < 0        : {(df_daily['Ventas'] < 0).sum()}  (devoluciones > ventas ese día)")
print(f"\n  Columnas del dataframe diario: {list(df_daily.columns)}")
print(f"\n  Primeras filas:")
print(df_daily.head(10).to_string(index=False))
print(f"\n  Estadísticas de 'Ventas' (£) — venta neta diaria:")
print(df_daily['Ventas'].describe().apply(lambda x: f"    {x:>12,.2f}").to_string())
print(f"\n  ✓ df_daily creado con {len(df_daily)} filas. Listo para las transformaciones siguientes.")


  Rango temporal             : 2010-12-01 → 2011-12-09
  Días en el rango completo  : 374
  Días con Ventas > 0        : 305
  Días con Ventas = 0 (hueco): 69
  Días con Ventas < 0        : 0  (devoluciones > ventas ese día)

  Columnas del dataframe diario: ['Fecha', 'Ventas', 'NumTransacc', 'NumPedidos', 'NumClientes', 'UnidadesVendidas']

  Primeras filas:
     Fecha   Ventas  NumTransacc  NumPedidos  NumClientes  UnidadesVendidas
2010-12-01 51118.57       3020.0       127.0         95.0           23464.0
2010-12-02 40795.79       2020.0       141.0         98.0           22960.0
2010-12-03 41801.33       2123.0        68.0         50.0           14596.0
2010-12-04     0.00          0.0         0.0          0.0               0.0
2010-12-05 29650.64       2591.0        88.0         75.0           14939.0
2010-12-06 47492.93       3757.0       102.0         82.0           20048.0
2010-12-07 57696.38       2835.0        82.0         65.0           18424.0
2010-12-08 40935.64       2519

In [ ]:
# [DOC] generamos variables temporales básicas, variables de memoria y ventanas móviles

# [DOC] Variables Temporales Básicas
df_daily_sales['DayOfWeek'] = df_daily_sales['Date'].dt.dayofweek
df_daily_sales['Month'] = df_daily_sales['Date'].dt.month
df_daily_sales['IsWeekend'] = df_daily_sales['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# [DOC] Variables de Memoria (Lags)
# No solo el día anterior (Lag 1), sino el mismo día de la semana pasada (Lag 7)
df_daily_sales['Sales_Lag1'] = df_daily_sales['Sales'].shift(1)
df_daily_sales['Sales_Lag7'] = df_daily_sales['Sales'].shift(7)

# [DOC] Ventanas Móviles (Rolling Windows)
# Promedio de ventas de los últimos 7 días (suaviza picos aislados)
df_daily_sales['Rolling_Mean_7'] = df_daily_sales['Sales'].shift(1).rolling(window=7).mean()

df_daily_sales.dropna(inplace=True)

In [ ]:
# [DOC] Aplicamos One-Hot Encoding para el día de la semana y separamos el conjunto de entrenamiento y test antes de escalar

df_daily_sales = pd.get_dummies(df_daily_sales, columns=['DayOfWeek'], prefix='Day')

split_date = pd.to_datetime('2011-11-09')

df_train = df_daily_sales[df_daily_sales['Date'] < split_date].copy()
df_test = df_daily_sales[df_daily_sales['Date'] >= split_date].copy()

print(f"[INFO] Registros Entrenamiento: {len(df_train)}")
print(f"[INFO] Registros Test: {len(df_test)}")

In [ ]:
# [DOC] Escalamos las variables numéricas para que todas tengan el mismo peso en el modelo.

features_to_scale = ['Sales_Lag1', 'Sales_Lag7', 'Rolling_Mean_7', 'TransactionCount']
scaler = StandardScaler()

df_train[features_to_scale] = scaler.fit_transform(df_train[features_to_scale])
df_test[features_to_scale] = scaler.transform(df_test[features_to_scale])

In [ ]:
# [DOC] Realizamos el guardado de los datos procesados

output_dir = '../../data/processed'

if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f"[INFO] Carpeta creada con éxito: {output_dir}")

df_train.to_csv(os.path.join(output_dir, 'train_data.csv'), index=False)
df_test.to_csv(os.path.join(output_dir, 'test_data.csv'), index=False)

print(f"[SUCCESS] Archivos guardados correctamente en: {output_dir}")
display(df_train.head())